# #22B — FDA Adverse Drug Event Analysis (Pharmacovigilance) — Build Notebook

Real, raw FAERS (FDA Adverse Event Reporting System) data — all of 2024 (Q1-Q4), downloaded directly from
FDA's own servers, not the openFDA API's JSON wrapper. Two deliberate design choices baked into this build
(see `PROJECT_NARRATIVE.md` Section 5 for the full reasoning):

1. **All four drug roles** (primary suspect / secondary suspect / concomitant / interacting) are used in the
   disproportionality signal-detection step, not the standard suspect-only (PS+SS) convention — a disclosed
   departure from typical pharmacovigilance practice.
2. **The fetch step is WRITE-IT, not GIVEN** — you are downloading the real files yourself from FDA's
   servers. Only the generic HTTP-download/unzip mechanics are boilerplate; the actual per-table loading
   (correct delimiter, correct dtypes, which columns matter) is yours to write from the real schema.

~80-90% of this notebook is blank (`____`) for you to fill in from the hint above each cell + the linked
docs. Every blank has the exact, correct real code as a trailing comment.

## Step 0 — Fetch the real FAERS 2024 data

**WHAT:** Download and unzip all four 2024 quarterly FAERS ASCII extracts from FDA's own export servers.

**WHY:** This is the real, raw pharmacovigilance data — the same files a working drug-safety data scientist
would pull. Downloading it yourself (rather than being handed a pre-fetched file) is deliberate, per your own
instruction on #18: you don't learn data engineering if someone else always does the fetching for you.

**HOW:** FDA publishes each quarter as a zip at a predictable URL:
`https://fis.fda.gov/content/Exports/faers_ascii_2024q{N}.zip` for N in 1,2,3,4. Use Python's `requests` to
download each, then `zipfile.ZipFile` to extract. This can take a while (each zip is several hundred MB) —
run it once, then reuse the extracted files (don't re-download on every notebook rerun).

In [ ]:
# ============================================================
# GIVEN — imports, directory setup, and a GENERIC, reusable download+unzip
# helper. This helper knows nothing FAERS-specific — it just downloads a URL
# to a file and unzips it if requested. True boilerplate, safe to reuse for
# any future project that needs to fetch+unzip a file.
# ============================================================
import os
import requests
import zipfile
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, classification_report

DATA_DIR = "data_py"
RESULTS_DIR = "results_py"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

def download_and_unzip(url, dest_dir, zip_name):
    """GENERIC helper: download `url` to dest_dir/zip_name, then unzip it into dest_dir.
    Skips the download if the zip already exists locally (don't hammer FDA's servers on rerun)."""
    zip_path = os.path.join(dest_dir, zip_name)
    if not os.path.exists(zip_path):
        print(f"downloading {url} ...")
        r = requests.get(url, stream=True, timeout=120)
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    else:
        print(f"{zip_path} already downloaded, skipping fetch")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest_dir)
    print(f"extracted to {dest_dir}")


### Write it — download all four 2024 quarters

**WHAT:** Build the 4 real FDA export URLs (one per quarter) and call `download_and_unzip` on each.

**WHY:** Each quarter is a separate zip on FDA's servers; you need all four for a full-year analysis.

**HOW:** The URL pattern is `https://fis.fda.gov/content/Exports/faers_ascii_2024q{N}.zip` — build a dict
mapping quarter label (e.g. `"2024_Q1"`) to its URL, then loop and call the GIVEN helper for each.

In [ ]:
quarters = {"2024_Q1": "https://fis.fda.gov/content/Exports/faers_ascii_2024q1.zip", "2024_Q2": "https://fis.fda.gov/content/Exports/faers_ascii_2024q2.zip", "2024_Q3": "https://fis.fda.gov/content/Exports/faers_ascii_2024q3.zip", "2024_Q4": "https://fis.fda.gov/content/Exports/faers_ascii_2024q4.zip"}

for qname, qurl in quarters.items():
    qdir = os.path.join(DATA_DIR, qname)
    os.makedirs(qdir, exist_ok=True)
    download_and_unzip(qurl, qdir, f"{qname}.zip")


### Write it — find each quarter's ASCII table files

**WHAT:** After unzipping, each quarter has an `ASCII/` subfolder containing `DEMO24Q{N}.txt`,
`DRUG24Q{N}.txt`, `REAC24Q{N}.txt`, `OUTC24Q{N}.txt` (THER/INDI/RPSR also exist but aren't needed for this
project's signal-detection + seriousness-classifier scope).

**WHY:** FDA's zip structure nests the actual files a couple of directories deep — you need the real path
before you can load anything.

**HOW:** `glob` for files matching each pattern inside every quarter's extracted folder; print what you find
to confirm before loading (a WRITE-IT step should always be checked against reality, not assumed).

In [ ]:
import glob

def find_table_files(table_prefix):
    """Return a sorted list of file paths for a given table across all 4 quarters,
    e.g. find_table_files("DEMO") -> [.../DEMO24Q1.txt, .../DEMO24Q2.txt, ...]"""
    pattern = os.path.join(DATA_DIR, "*", "ASCII", f"{table_prefix}24Q*.txt")
    return sorted(glob.glob(pattern))

demo_files = find_table_files("DEMO")
drug_files = find_table_files("DRUG")
reac_files = find_table_files("REAC")
outc_files = find_table_files("OUTC")
print(demo_files, drug_files, reac_files, outc_files)


## Step 1 — Load and concatenate each table across all 4 quarters

**WHAT:** Read each quarter's pipe(`$`)-delimited file for DEMO, DRUG, REAC, OUTC, and stack the 4 quarters
of each table into one DataFrame.

**WHY:** The signal-detection and classifier steps both need the FULL year's cases, not one quarter at a
time — the whole point of using the raw files was full-year coverage.

**HOW:** `pd.read_csv(path, sep="$", dtype=str, encoding="latin1", low_memory=False)` — real FAERS gotchas:
the delimiter is a literal `$`, NOT a comma; force `dtype=str` for now (many "numeric-looking" columns like
`primaryid`/`caseid` should stay strings — pandas will otherwise guess wrong types across files with missing
values, which corrupts joins later); FAERS files aren't guaranteed UTF-8 (some free-text/company-name fields
use Latin-1 characters), so `encoding="latin1"` avoids a decode crash partway through a 60MB+ file.

In [ ]:
def load_and_concat(file_list):
    """Read every file in file_list as a pipe-delimited table, dtype=str, latin1 encoding,
    and concatenate them into one DataFrame."""
    frames = []
    for path in file_list:
        df = pd.read_csv(path, sep="$", dtype=str, encoding="latin1", low_memory=False)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

demo = load_and_concat(demo_files)
drug = load_and_concat(drug_files)
reac = load_and_concat(reac_files)
outc = load_and_concat(outc_files)
print("raw row counts:", len(demo), len(drug), len(reac), len(outc))


## Step 2 — Deduplicate: keep only each case's latest caseversion

**WHAT:** For every unique `caseid` in `demo`, keep only the row with the MAXIMUM `caseversion` (an amended
case gets a new, higher caseversion under the same caseid, sometimes in a later quarter's file).

**WHY:** Without this, a case resubmitted after Q1 (appearing again, updated, in a later quarter) gets
counted more than once — silently inflating both the drug-reaction signal counts and the classifier's row
count with duplicated case data. This is a real, FAERS-specific gotcha named explicitly in WORKFLOW.md.

**HOW:** `caseversion` arrived as a string (we forced `dtype=str` above) — cast it to numeric first. Then
group by `caseid`, find the max caseversion per group, and keep only rows matching that max. Once you have
the deduplicated set of `(caseid, caseversion)` pairs that should survive, filter DRUG/REAC/OUTC down to only
the `primaryid`s belonging to those surviving (caseid, caseversion) pairs too — every table must agree on
which case-version is "the real one" for that case.

In [ ]:
demo["caseversion"] = pd.to_numeric(demo["caseversion"], errors="coerce")

# for each caseid, find the row(s) with the max caseversion
latest_idx = demo.groupby("caseid")["caseversion"].idxmax()
demo_dedup = demo.loc[latest_idx].reset_index(drop=True)

print(f"demo: {len(demo)} raw rows -> {len(demo_dedup)} deduplicated cases")

# the set of primaryids that belong to a surviving (caseid, caseversion)
keep_primaryids = set(demo_dedup["primaryid"])

drug_dedup = drug[drug["primaryid"].isin(keep_primaryids)].reset_index(drop=True)
reac_dedup = reac[reac["primaryid"].isin(keep_primaryids)].reset_index(drop=True)
outc_dedup = outc[outc["primaryid"].isin(keep_primaryids)].reset_index(drop=True)

print(f"drug: {len(drug)} -> {len(drug_dedup)} | reac: {len(reac)} -> {len(reac_dedup)} | outc: {len(outc)} -> {len(outc_dedup)}")


## Step 3 — Build the SERIOUS label from OUTC

**WHAT:** For every deduplicated case (`caseid`), determine whether it has at least one row in
`outc_dedup` — FAERS' own regulatory definition of a serious report (death/life-threatening/hospitalization/
disability/congenital anomaly/other-important-event, whichever specific code(s) apply).

**WHY:** This is a clean, regulation-grounded label derived directly from the raw schema, rather than trusted
from a pre-computed field someone else built — you know exactly what it means and can explain it to an
interviewer.

**HOW:** Get the set of `primaryid`s present in `outc_dedup` (any row = serious), then build a `serious`
column on `demo_dedup` that's 1 if the case's `primaryid` is in that set, else 0.

In [ ]:
serious_primaryids = set(outc_dedup["primaryid"])

demo_dedup["serious"] = demo_dedup["primaryid"].isin(serious_primaryids).astype(int)

print(demo_dedup["serious"].value_counts())
print("fraction serious:", demo_dedup["serious"].mean())


## Step 4 — Build per-drug and per-reaction CASE counts

**WHAT:** For a manageable, disclosed shortlist (the top-N most-reported drugs and top-N most-reported
reactions), count how many DISTINCT deduplicated cases mention each drug, and how many mention each reaction.

**WHY:** PRR/ROR need case-level presence/absence, not raw row counts — a case listing the same drug twice
(e.g. two doses/routes) must count as ONE case having that drug, not two. This project's scope (Section 5 of
PROJECT_NARRATIVE.md) uses ALL drug roles (PS/SS/C/I), not just suspect drugs — don't filter `role_cod` here.

**HOW:** `drugname` needs light cleaning (uppercase, strip whitespace — the same real drug shows up with
inconsistent casing/spacing across independently-submitted reports). Group by cleaned drug name, count
NUNIQUE `caseid`s (not rows). Same idea for `reac_dedup`'s `pt` column. Keep only the top ~150 drugs and top
~150 reactions by case count, to keep the pairwise signal computation in the next step tractable.

In [ ]:
drug_dedup["drugname_clean"] = drug_dedup["drugname"].str.upper().str.strip()

drug_case_counts = drug_dedup.groupby("drugname_clean")["caseid"].nunique().sort_values(ascending = False)
reac_case_counts = reac_dedup.groupby("pt")["caseid"].nunique().sort_values(ascending = False)

TOP_N = 150
top_drugs = drug_case_counts.head(TOP_N).index.tolist()
top_reactions = reac_case_counts.head(TOP_N).index.tolist()

print(f"top drug: {top_drugs[0]} ({drug_case_counts.iloc[0]} cases)")
print(f"top reaction: {top_reactions[0]} ({reac_case_counts.iloc[0]} cases)")


## Step 5 — Compute PRR, ROR, and chi-square for each drug x reaction pair

**WHAT:** For every (drug, reaction) pair among your top-N shortlists, build the 2x2 contingency table
(a = both present, b = drug only, c = reaction only, d = neither) over the FULL deduplicated case set, then
compute PRR, ROR, and chi-square from it.

**WHY:** This is the actual disproportionality-analysis math from WORKFLOW.md Section 2 — the core scientific
step of the whole project. Doing it from the raw 2x2 counts (rather than a black-box library call) means you
can explain exactly what each number means.

**HOW:** You need, per case, the SET of drugs it has and the SET of reactions it has (a case can have several
of each) — build these as two dicts mapping `caseid -> set(...)` once, then reuse them for every pair rather
than re-scanning the full table per pair (that would be far too slow at this scale). `a` = cases with both
drug in its drug-set AND reaction in its reaction-set; `b` = cases with the drug but not the reaction; `c` =
reaction but not drug; `d` = total cases minus (a+b+c). PRR = (a/(a+b)) / (c/(c+d)); ROR = (a*d)/(b*c);
chi-square via `scipy.stats.chi2_contingency([[a,b],[c,d]])`. Apply a minimum case-count floor (a >= 3) before
trusting a pair's ratio, per the field's own convention (see WORKFLOW.md point 4 of "things that will bite").

In [ ]:
# filter to the top-N shortlist FIRST -- building a per-case set over the full
# multi-million-row table (when only ~150 drugs/reactions ever get used below)
# is what causes an out-of-memory kernel crash at this scale.
drug_relevant = drug_dedup[drug_dedup["drugname_clean"].isin(top_drugs)]
reac_relevant = reac_dedup[reac_dedup["pt"].isin(top_reactions)]

case_drugs = drug_relevant.groupby("caseid")["drugname_clean"].apply(set).to_dict()
case_reacs = reac_relevant.groupby("caseid")["pt"].apply(set).to_dict()

total_cases = demo_dedup["caseid"].nunique()

drug_case_sets = {d: set() for d in top_drugs}
for cid, dset in case_drugs.items():
    for d in dset:
        if d in drug_case_sets:
            drug_case_sets[d].add(cid)

reac_case_sets = {r: set() for r in top_reactions}
for cid, rset in case_reacs.items():
    for r in rset:
        if r in reac_case_sets:
            reac_case_sets[r].add(cid)

MIN_CASES = 3
signal_rows = []
for d in top_drugs:
    d_cases = drug_case_sets[d]
    for r in top_reactions:
        r_cases = reac_case_sets[r]
        a = len(d_cases & r_cases)
        if a < MIN_CASES:
            continue
        b = len(d_cases - r_cases)
        c = len(r_cases - d_cases)
        d_cell = total_cases - a - b - c
        prr = (a/(a+b))/(c/(c + d_cell)) if (c > 0 and (c + d_cell) > 0) else np.nan  # (a / (a + b)) / (c / (c + d_cell)) if (c > 0 and (c + d_cell) > 0) else np.nan
        ror = (a*d_cell)/(b*c) if (b > 0 and c > 0) else np.nan  # (a * d_cell) / (b * c) if (b > 0 and c > 0) else np.nan
        chi2, pval, _, _ = stats.chi2_contingency([[a,b],[c, d_cell]])
        signal_rows.append({"drug": d, "reaction": r, "a": a, "b": b, "c": c, "d": d_cell,
                             "PRR": prr, "ROR": ror, "chi2": chi2, "pvalue": pval})

signal_df = pd.DataFrame(signal_rows).sort_values("PRR", ascending=False)
signal_df.head(20)


## Step 6 — Build the seriousness-classifier feature table

**WHAT:** One row per deduplicated case: age, sex, total drug count, count of drugs per role (PS/SS/C/I),
total distinct reaction count — merged with the `serious` label from Step 3.

**WHY:** These are the only features available in the report's own structure (no clinical judgment data
exists in FAERS) — the whole point of Step 6/7 is testing whether structural shape alone predicts
seriousness.

**HOW:** Per-case drug-role counts: pivot `drug_dedup` on `caseid` x `role_cod`, counting rows, then reindex
to make sure all 4 role columns (PS/SS/C/I) exist even if a case has zero of a given role. Per-case reaction
count: `reac_dedup.groupby("caseid").size()`. `age`/`sex` come straight from `demo_dedup` — cast `age` to
numeric (it arrived as a string), leaving unparseable values as NaN (some ages are recorded in the wrong
units or missing entirely — a real messiness of self/clinician-reported data, don't silently drop these rows,
let the model/imputation handle NaN explicitly).

In [ ]:
role_counts = drug_dedup.pivot_table(index="caseid", columns="role_cod", aggfunc="size", fill_value=0)
role_counts = role_counts.reindex(columns=["PS", "SS", "C", "I"], fill_value=0)
role_counts.columns = [f"n_role_{c}" for c in role_counts.columns]

reaction_counts = reac_dedup.groupby("caseid").size().rename("n_reactions")

features = demo_dedup[["caseid", "age", "sex", "serious"]].copy()
features["age"] = pd.to_numeric(features["age"], errors= "coerce")

features = features.merge(role_counts, on="caseid", how="left").merge(reaction_counts, on="caseid", how="left")
features[[c for c in features.columns if c.startswith("n_")]] = features[[c for c in features.columns if c.startswith("n_")]].fillna(0)

features["n_drugs_total"] = features[["n_role_PS", "n_role_SS", "n_role_C", "n_role_I"]].sum(axis=1)

print(features.shape)
features.head()


## Step 7 — Train/test split, Logistic Regression + Gradient Boosting

**WHAT:** Split `features` into train/test, encode `sex`, fit a logistic regression (interpretable baseline)
and a gradient boosting classifier (captures non-linear feature interactions) to predict `serious`, and
evaluate both on ROC-AUC and precision/recall.

**WHY:** Real class imbalance is expected (see WORKFLOW.md Section 2) — accuracy alone would be misleading
(a model that always predicts the majority class scores high accuracy but zero real skill), so
`class_weight="balanced"` and ROC-AUC/precision-recall are the honest metrics here.

**HOW:** Drop rows with missing `age`/`sex` for this first pass (or encode "missing" as its own category —
your call, document which you chose). One-hot or label-encode `sex`. `train_test_split(..., stratify=y)` to
keep the same serious/not-serious ratio in both splits. `LogisticRegression(class_weight="balanced")` and
`GradientBoostingClassifier()` (gradient boosting doesn't take `class_weight` directly — consider a
`sample_weight` argument in `.fit()` if you want to weight it too).

In [ ]:
model_df = features.dropna(subset=["age", "sex"]).copy()
model_df["sex_encoded"] = model_df["sex"].map({"M":0, "F":1, "UNK":2}).fillna(2)

feature_cols = ["age", "sex_encoded", "n_role_PS", "n_role_SS", "n_role_C", "n_role_I", "n_reactions", "n_drugs_total"]
X = model_df[feature_cols]
y = model_df["serious"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

logreg = LogisticRegression(class_weight="balanced", max_iter=1000)
logreg.fit(X_train, y_train)
logreg_proba = logreg.predict_proba(X_test)[:,1]

gbm = GradientBoostingClassifier(random_state=42)
gbm.fit(X_train, y_train)
gbm_proba = gbm.predict_proba(X_test)[:,1]

results_df = pd.DataFrame([
    {"model": "LogReg", "auc": roc_auc_score(y_test, logreg_proba)},
    {"model": "GradientBoosting", "auc": roc_auc_score(y_test, gbm_proba)},
])
results_df


## Step 8 — Feature importance / coefficients

**WHAT:** Extract logistic regression coefficients and gradient boosting feature importances, ranked.

**WHY:** This answers the actual research question for the classifier half of the project: WHICH structural
report features carry predictive signal about seriousness, and does that make biological/reporting sense
(e.g. more distinct reactions per case plausibly tracking a more complicated, more serious clinical picture)?

**HOW:** `logreg.coef_[0]` paired with `feature_cols`; `gbm.feature_importances_` the same way; sort both
descending by absolute value / raw value respectively.

In [ ]:
logreg_importance = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": logreg.coef_[0],
}).sort_values("coefficient", key=abs, ascending=False)

gbm_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": gbm.feature_importances_,
}).sort_values("importance", ascending=False)

print(logreg_importance)
print(gbm_importance)


## Step 9 — Export results

**WHAT:** Save `signal_df`, `results_df`, `logreg_importance`, `gbm_importance`, and the top drug/reaction
case-count tables as CSVs in `results_py/`, plus the deduplicated `demo_dedup`/feature bridge files in
`data_py/` for the R twin to read.

**WHY:** Every other project in this portfolio exports real numbers as CSV, not just prints them — and the R
twin needs bridge files (this project's cross-language bridge, disclosed per house convention) to work from
the same deduplicated case data rather than re-implementing the fetch+dedup logic a second time. Two of
those bridge files (`case_drug_long.csv`, `case_reac_long.csv`) are case-level (caseid, drug)/(caseid,
reaction) rows, restricted to the top-N shortlists — R's PhViD package needs a pre-aggregated drug x reaction
count table (see WORKFLOW.md / 02_build.R Step 2), and these two files are what it builds that count table
from.

In [ ]:
signal_df.to_csv(os.path.join(RESULTS_DIR, "signal_scores.csv"), index=False)
drug_case_counts.head(50).to_csv(os.path.join(RESULTS_DIR, "top_drugs.csv"))
reac_case_counts.head(50).to_csv(os.path.join(RESULTS_DIR, "top_reactions.csv"))
results_df.to_csv(os.path.join(RESULTS_DIR, "model_metrics.csv"), index=False)
logreg_importance.to_csv(os.path.join(RESULTS_DIR, "feature_importance_logreg.csv"), index=False)
gbm_importance.to_csv(os.path.join(RESULTS_DIR, "feature_importance_gbm.csv"), index=False)

# bridge files for the R twin
model_df.to_csv(os.path.join(DATA_DIR, "model_features.csv"), index=False)

# two more R-twin bridge files: case-level (caseid, drug) and (caseid, reaction)
# rows, restricted to the same top-N shortlists Python used, so R's PhViD step
# builds its drug x reaction count table from the identical underlying data
case_drug_long = pd.DataFrame(
    [(cid, d) for cid, dset in case_drugs.items() for d in dset if d in drug_case_sets],
    columns=["caseid", "drug"]
)
case_reac_long = pd.DataFrame(
    [(cid, r) for cid, rset in case_reacs.items() for r in rset if r in reac_case_sets],
    columns=["caseid", "reaction"]
)
case_drug_long.to_csv(os.path.join(DATA_DIR, "case_drug_long.csv"), index=False)
case_reac_long.to_csv(os.path.join(DATA_DIR, "case_reac_long.csv"), index=False)

# third R-twin bridge file: the true full deduplicated case count, so R's
# PRR/ROR denominator matches this same total_cases used above (Step 5) --
# NOT nrow(model_features), which is narrowed by dropna(age, sex) for the
# classifier and is the wrong denominator for signal detection.
pd.DataFrame({"total_cases": [total_cases]}).to_csv(os.path.join(DATA_DIR, "total_cases.csv"), index=False)

print("done")
